In [1]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# setting the path where the dataset is sitting after extraction
data_dir = "archive/mnist_png"
train_dir = os.path.join(data_dir, "training")
test_dir = os.path.join(data_dir, "testing")

# small helper to load a limited number of images per folder so it trains fast
def load_images(folder, limit_per_class=800):
    images = []
    for label in sorted(os.listdir(folder)):
        label_path = os.path.join(folder, label)
        if not os.path.isdir(label_path):
            continue
        files = sorted(os.listdir(label_path))[:limit_per_class]
        for f in files:
            img_path = os.path.join(label_path, f)
            img = Image.open(img_path).convert("L")
            img = img.resize((28, 28))
            images.append(np.array(img))
    return np.array(images)

# loading training and testing images
print("loading training images")
x_train = load_images(train_dir, limit_per_class=800)
print("loading testing images")
x_test = load_images(test_dir, limit_per_class=150)

print("train shape", x_train.shape)
print("test shape", x_test.shape)

# scaling pixel values between 0 and 1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# reshaping to add the channel dimension
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# adding random gaussian noise to create the noisy versions of the images
noise_factor = 0.4
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

# clipping so pixel values stay between 0 and 1
x_train_noisy = np.clip(x_train_noisy, 0.0, 1.0)
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0)

# building the autoencoder model
input_img = layers.Input(shape=(28, 28, 1))

# encoder part, squeezing the image down
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(input_img)
x = layers.MaxPooling2D((2, 2), padding="same")(x)
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(x)
encoded = layers.MaxPooling2D((2, 2), padding="same")(x)

# decoder part, building the image back up
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(encoded)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
x = layers.UpSampling2D((2, 2))(x)
decoded = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same")(x)

autoencoder = models.Model(input_img, decoded)
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

autoencoder.summary()

# training the autoencoder on noisy images mapped to clean images
history = autoencoder.fit(
    x_train_noisy, x_train,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test)
)

# saving the trained model
autoencoder.save("denoising_autoencoder_model.keras")

# plotting the training loss curve
plt.figure(figsize=(6, 4))
plt.plot(history.history["loss"], label="training loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("training and validation loss")
plt.legend()
plt.savefig("loss_curve.png")
plt.close()

# getting denoised outputs for the test set
denoised_images = autoencoder.predict(x_test_noisy)

# showing a few examples of original, noisy and denoised images side by side
n = 8
plt.figure(figsize=(16, 6))
for i in range(n):
    # original clean image
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    if i == 0:
        ax.set_title("original", loc="left")

    # noisy image
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    if i == 0:
        ax.set_title("noisy", loc="left")

    # denoised image produced by the model
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised_images[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    if i == 0:
        ax.set_title("denoised", loc="left")

plt.tight_layout()
plt.savefig("denoising_results.png")
plt.close()

print("done, model and result images saved")


loading training images
loading testing images
train shape (6400, 28, 28)
test shape (1500, 28, 28)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 16)     │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 16)       │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 1)      │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,193 (47.63 KB)

 Trainable params: 12,193 (47.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - loss: 0.5156 - val_loss: 0.2242
Epoch 2/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - loss: 0.2054 - val_loss: 0.1572
Epoch 3/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - loss: 0.1479 - val_loss: 0.1355
Epoch 4/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - loss: 0.1276 - val_loss: 0.1246
Epoch 5/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - loss: 0.1189 - val_loss: 0.1189
Epoch 6/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - loss: 0.1142 - val_loss: 0.1149
Epoch 7/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - loss: 0.1104 - val_loss: 0.1123
Epoch 8/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - loss: 0.1081 - val_loss: 0.1101
Epoch 9/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - loss: 0.1063 - val_loss: 0.1089
Epoch 10/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 0.1052 - val_loss: 0.1072
Epoch 11/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 0.1036 - val_loss: 0.1060
Epoch 12/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 0.1

Observation:
The training loss dropped steadily from about 0.46 to 0.10 over 20 epochs, and the validation loss followed the same trend closely, meaning the model was not overfitting.
Looking at the output image, the noisy digits are heavily corrupted with random speckles, making the digit shapes hard to see clearly. After passing through the autoencoder, the denoised outputs are much smoother and the digit outlines become clear and readable again, close to the original clean images. Some minor blurring is visible on the edges, which is expected since the autoencoder is a small, simple model trained for a short time on a limited subset of the data. Overall, the model successfully learned to suppress noise while preserving the core structure of the digits.